In [14]:
import torch
import torch.nn as nn 
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

import pandas as pd 
import numpy as np 

import re

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter(
    log_dir="runs/word2vec"
)


from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

[nltk_data] Downloading package punkt to /home/eshaan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/eshaan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/eshaan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


device(type='cuda')

In [ ]:
vocab_size = 5000

def encoder(path, vocab_limit) : 

    folder_path = Path(path)
    txt_files = sorted(folder_path.glob("*.txt"))   


    word_tokenised_sentences = []
    all_words = []
    encoded_sentences = []
    
    for file_path in txt_files:
        
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()
            text = re.sub(r"\s+", " ", text).strip()
            sentences = sent_tokenize(text.lower())

        stop_words = set(stopwords.words("english"))


        for sentence in sentences : 
            word_tokenised_sentence = []
            for word in word_tokenize(sentence):
                if word not in stop_words and word.isalpha():
                    word_tokenised_sentence.append(word)
                    all_words.append(word)
                    
            word_tokenised_sentences.append(word_tokenised_sentence)

        
    all_words = pd.Series(all_words)
    freq_words = all_words.value_counts().head(vocab_limit -1).index.to_list()
    freq_words = ["<UNK>"] + freq_words

    word_index_dict = {word : idx for idx,word in enumerate(freq_words)}

    for sentence in word_tokenised_sentences :
        encoded_sentence = []
        for word_token in sentence:
            encoded_sentence.append(
                word_index_dict.get(word_token,word_index_dict["<UNK>"])
            )
        encoded_sentences.append(encoded_sentence)

    return word_index_dict, encoded_sentences    


word_index_dict, encoded_sentences = encoder(
    r"../datasets/hp_books",
    vocab_size
)

# encoded_sentences

In [20]:
len(encoded_sentences)

6515

In [3]:
class Cbag_Dataset(Dataset):

    def __init__(self, encoded_sentences, window_size = 4):

        self.samples = []

        for sentence in encoded_sentences :
            for i in range(window_size, len(sentence) - window_size) : 
                target_word = torch.tensor(
                    sentence[i]
                    )

                context_words = torch.tensor(
                    sentence[ i-window_size : i ] + sentence[ i+1  :  i+window_size+1 ]
                    ) 

                self.samples.append((context_words, target_word))

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        context, target = self.samples[idx]
        return (
            torch.tensor(context, dtype=torch.long),
            torch.tensor(target, dtype=torch.long)
        )


In [4]:
dataset = Cbag_Dataset(encoded_sentences,4)
print(len(dataset))

train_dataloader = DataLoader(
    dataset=dataset,
    batch_size=32,
    pin_memory=True,
    shuffle=True,
)


9364


In [5]:
class simpleW2V(nn.Module):

    def __init__(self, vocab_count, feature_count):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings = vocab_count,
            embedding_dim = feature_count,
            device= device
        )

        self.output = nn.Linear(
            in_features=feature_count,
            out_features=vocab_count,
            device= device
        )


    def forward(self, x):
        # x shape = (batch_size, context_words)
        
        features_x = self.embedding(x)   # shape = (batch_size, context_words, feature_count)
        features_x = features_x.mean(dim = 1)         # shape = (batch_size, feature_count)
        logits  = self.output(features_x)

        
        return logits 

model = simpleW2V(vocab_size, 300)
model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [6]:
print("Vocabulary:", len(word_index_dict))
print("Sentences:", len(encoded_sentences))
print("Training samples:", len(dataset))
print("Batches:", len(train_dataloader))

contexts, targets = next(iter(train_dataloader))

print("Contexts:", contexts.shape)
print("Targets:", targets.shape)

Vocabulary: 5000
Sentences: 6515
Training samples: 9364
Batches: 293
Contexts: torch.Size([32, 8])
Targets: torch.Size([32])


/tmp/ipykernel_27456/2565846015.py:25: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(context, dtype=torch.long),
/tmp/ipykernel_27456/2565846015.py:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(target, dtype=torch.long)


In [7]:
epochs = 300 
global_step = 0

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for context_words, target_word in train_dataloader:

        context_words = context_words.to(device)
        target_word = target_word.to(device)

        logits = model(context_words)

        # print("context_words:", context_words.shape)
        # print("logits:", logits.shape)
        # print("target_word:", target_word.shape)

        loss = criterion(logits, target_word)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        writer.add_scalar(
            "Loss/Batch",
            loss.item(),
            global_step
        )
        global_step+=1


    avg_loss = total_loss / len(train_dataloader)


    writer.add_scalar(
        "Loss/Epoch",
        avg_loss,
        epoch
    )


    print(f"EPOCH : {epoch}, avg-loss : {avg_loss}")

/tmp/ipykernel_27456/2565846015.py:25: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(context, dtype=torch.long),
/tmp/ipykernel_27456/2565846015.py:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(target, dtype=torch.long)


EPOCH : 0, avg-loss : 7.6558266060344184
EPOCH : 1, avg-loss : 6.233723272642584
EPOCH : 2, avg-loss : 5.350204809533858
EPOCH : 3, avg-loss : 4.538929639009079
EPOCH : 4, avg-loss : 3.7483873009274844
EPOCH : 5, avg-loss : 3.0086188593008414
EPOCH : 6, avg-loss : 2.376383309478239
EPOCH : 7, avg-loss : 1.8785775785966945
EPOCH : 8, avg-loss : 1.495908676967686
EPOCH : 9, avg-loss : 1.1982062060271514
EPOCH : 10, avg-loss : 0.9612634535942468
EPOCH : 11, avg-loss : 0.7709442743262334
EPOCH : 12, avg-loss : 0.6190487662069627
EPOCH : 13, avg-loss : 0.49824538450599126
EPOCH : 14, avg-loss : 0.4006706402977172
EPOCH : 15, avg-loss : 0.3228874320463109
EPOCH : 16, avg-loss : 0.2614772673759851
EPOCH : 17, avg-loss : 0.21231771263042815
EPOCH : 18, avg-loss : 0.17339872462045616
EPOCH : 19, avg-loss : 0.14232606260239467
EPOCH : 20, avg-loss : 0.11735807420247243
EPOCH : 21, avg-loss : 0.0972146724601858
EPOCH : 22, avg-loss : 0.08094424885338483
EPOCH : 23, avg-loss : 0.06761674495229542


In [ ]:
def get_weight_vector(type, idx):
    if type == 'in' :
        w_matrix = model.embedding.weight.detach().cpu()
    else:
        w_matrix = model.output.weight.detach().cpu()

    return w_matrix[idx]


def similarity(type,word1, word2) :
    word1_idx = word_index_dict[word1]
    word2_idx = word_index_dict[word2]
    w1 = get_weight_vector(type ,word1_idx)
    w2 = get_weight_vector(type ,word2_idx)

    return F.cosine_similarity(
        w1.unsqueeze(0),
        w2.unsqueeze(0)
    ).item()




def most_similar(word, matrix_type="in", top_k=10):

    if word not in word_index_dict:
        raise ValueError(f"{word} is not in vocabulary")

    if matrix_type == "in":
        W = model.embedding.weight.detach().cpu()

    elif matrix_type == "out":
        W = model.output.weight.detach().cpu()

    else:
        raise ValueError("matrix_type must be 'in' or 'out'")

    target_idx = word_index_dict[word]
    target_vector = W[target_idx]

    similarities = F.cosine_similarity(
        W,
        target_vector.unsqueeze(0),
        dim=1
    )

    values, indices = torch.topk(
        similarities,
        k=top_k + 1
    )

    idx_to_word = {
        idx: word
        for word, idx in word_index_dict.items()
    }

    results = []

    for score, idx in zip(values, indices):

        candidate_word = idx_to_word[idx.item()]

        # skip the word itself
        if candidate_word == word:
            continue

        results.append(
            (candidate_word, score.item())
        )

        if len(results) == top_k:
            break

    return results

In [12]:
word_index_dict

{'<UNK>': 0,
 'harry': 1,
 'said': 2,
 'potter': 3,
 'ron': 4,
 'stone': 5,
 'hagrid': 6,
 'page': 7,
 'philosophers': 8,
 'rowling': 9,
 'hermione': 10,
 'back': 11,
 'one': 12,
 'got': 13,
 'could': 14,
 'get': 15,
 'like': 16,
 'know': 17,
 'see': 18,
 'professor': 19,
 'snape': 20,
 'looked': 21,
 'dumbledore': 22,
 'around': 23,
 'dudley': 24,
 'going': 25,
 'go': 26,
 'something': 27,
 'look': 28,
 'malfoy': 29,
 'never': 30,
 'right': 31,
 'think': 32,
 'uncle': 33,
 'yeh': 34,
 'time': 35,
 'neville': 36,
 'well': 37,
 'vernon': 38,
 'quirrell': 39,
 'first': 40,
 'would': 41,
 'door': 42,
 'even': 43,
 'eyes': 44,
 'looking': 45,
 'mcgonagall': 46,
 'head': 47,
 'two': 48,
 'people': 49,
 'thought': 50,
 'next': 51,
 'come': 52,
 'way': 53,
 'told': 54,
 'still': 55,
 'room': 56,
 'face': 57,
 'though': 58,
 'gryffindor': 59,
 'boy': 60,
 'good': 61,
 'last': 62,
 'left': 63,
 'us': 64,
 'behind': 65,
 'hogwarts': 66,
 'ter': 67,
 'house': 68,
 'turned': 69,
 'much': 70,
 'say

In [13]:
print(similarity("out", "professor", "dumbledore"))

-0.08601650595664978
